# PU Learning Experiments: Stability & Method Comparison

This notebook extends the main `lead_scoring.ipynb` with two thesis-strengthening experiments:

**Experiment A — Multi-seed stability.** Run Bagging-PU with 5 different random seeds and
report Recall@k as mean ± standard deviation. Directly answers: *"how reproducible are the
results — would you get the same numbers if you ran it again?"*

**Experiment B — Elkan-Noto comparison.** Implement the Elkan-Noto PU-correction method
(Elkan & Noto, 2008) as an alternative to Bagging-PU. Compare both on the same held-out
validation set. Shows Bagging-PU is a considered choice, not the only option tried.

**Prerequisite:** `features.csv` must exist in the same folder (produced by `feature_engineering.py`).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# Experiment settings -- keep these stable across A and B for a fair comparison.
N_ITERATIONS      = 50
NEGATIVE_MULTIPLE = 3
HOLDOUT_FRAC      = 0.20                    # positives held out for evaluation
SEEDS             = [42, 123, 7, 2024, 99]  # Experiment A: 5 seeds. Bump to 10 if you have time.
K_VALUES          = [100, 500, 1000, 2000, 5000]

## 2. Load features (same as main notebook)

In [ ]:
features = pd.read_csv("features.csv")
for c in ("category", "search_category", "governorate"):
    features[c] = features[c].astype("category")

y = features["is_customer"].values
X = features.drop(columns=["is_customer"])
print(f"Rows: {len(X):,}   Positives: {int(y.sum())}   Features: {X.shape[1]}")

## 3. Reusable functions

Wrapping Bagging-PU and Elkan-Noto so both experiments can call them cleanly.

In [ ]:
def make_lgb(seed: int) -> lgb.LGBMClassifier:
    """Standard LightGBM used inside every PU method here."""
    return lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, random_state=seed,
        n_jobs=-1, verbose=-1,
    )


def make_split(y: np.ndarray, seed: int):
    """Split positives into train (80%) and held-out validation (20%). Returns:
      train_pos, holdout_pos, y_train (labels for training-only), is_holdout (bool mask).
    Unlabeled points stay unlabeled in y_train; held-out positives are disguised as unlabeled."""
    pos_idx = np.where(y == 1)[0]
    train_pos, holdout_pos = train_test_split(pos_idx, test_size=HOLDOUT_FRAC, random_state=seed)
    y_train = np.zeros(len(y), dtype=int)
    y_train[train_pos] = 1
    is_holdout = np.zeros(len(y), dtype=bool)
    is_holdout[holdout_pos] = True
    return train_pos, holdout_pos, y_train, is_holdout


def run_bagging_pu(X, y, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Standard Bagging-PU: N_ITERATIONS rounds of {positives vs random pseudo-negatives},
    then average out-of-bag scores per point. Returns (pu_scores, is_holdout_mask)."""
    rng = np.random.RandomState(seed)
    train_pos, _, y_train, is_holdout = make_split(y, seed)
    unlabeled_pool  = np.where(y_train == 0)[0]
    neg_sample_size = NEGATIVE_MULTIPLE * len(train_pos)
    score_sum   = np.zeros(len(X))
    score_count = np.zeros(len(X), dtype=int)
    for _ in range(N_ITERATIONS):
        neg_sample = rng.choice(unlabeled_pool, size=neg_sample_size, replace=False)
        train_idx  = np.concatenate([train_pos, neg_sample])
        y_round    = np.concatenate([np.ones(len(train_pos)), np.zeros(neg_sample_size)]).astype(int)
        clf = make_lgb(seed)
        clf.fit(X.iloc[train_idx], y_round)
        oob_mask = np.ones(len(X), dtype=bool)
        oob_mask[neg_sample] = False
        oob_scores = clf.predict_proba(X[oob_mask])[:, 1]
        score_sum[oob_mask]   += oob_scores
        score_count[oob_mask] += 1
    return score_sum / score_count, is_holdout


def run_elkan_noto(X, y, seed: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Elkan-Noto (2008), estimator 'e1':
      1. Split training positives into 'labeled' (80%) and 'c-estimation' (20%).
      2. Train ONE classifier: labeled positives (y=1) vs entire unlabeled pool (y=0).
      3. c = mean predicted score on the held-out c-estimation positives  -- estimates P(labeled | positive).
      4. Calibrate: p(positive | x) = p(labeled | x) / c.
    Returns (calibrated_scores, is_holdout_mask, c)."""
    rng = np.random.RandomState(seed)
    train_pos, _, y_train, is_holdout = make_split(y, seed)

    # Further split training positives -> labeled for training + held-out for c-estimation
    labeled_pos, c_est_pos = train_test_split(train_pos, test_size=0.20, random_state=seed)

    y_naive = np.zeros(len(y), dtype=int)
    y_naive[labeled_pos] = 1                # only these are treated as positive
    # c_est_pos and evaluation holdout_pos both look like unlabeled (0) during training

    clf = make_lgb(seed)
    clf.fit(X, y_naive)
    raw_scores = clf.predict_proba(X)[:, 1]

    # c = average predicted score on the c-estimation positives (which the model was NOT told about)
    c = raw_scores[c_est_pos].mean()
    calibrated = np.clip(raw_scores / c, 0.0, 1.0)
    return calibrated, is_holdout, c


def precision_recall_at_k(scores: np.ndarray, is_target: np.ndarray, ks: list[int]) -> pd.DataFrame:
    order = np.argsort(-scores)
    hits_cum = np.cumsum(is_target[order])
    total_targets = is_target.sum()
    rows = []
    for k in ks:
        hits = int(hits_cum[k - 1])
        rows.append({"k": k, "hits": hits,
                     "precision@k": hits / k,
                     "recall@k": hits / total_targets})
    return pd.DataFrame(rows)

## 4. Experiment A: Multi-seed stability

Run Bagging-PU with each of `SEEDS` and record Recall@k. Report mean ± std across seeds.

**Expected runtime:** ~1–2 min per seed = ~5–10 min for 5 seeds.

In [ ]:
stability_records = []
for seed in SEEDS:
    print(f"Seed {seed}: running Bagging-PU ({N_ITERATIONS} iterations)...")
    scores, is_holdout = run_bagging_pu(X, y, seed)
    metrics = precision_recall_at_k(scores, is_holdout, K_VALUES)
    metrics["seed"] = seed
    stability_records.append(metrics)

stability_df = pd.concat(stability_records, ignore_index=True)

# Aggregate: mean ± std across seeds at each k
agg = stability_df.groupby("k").agg(
    recall_mean=("recall@k", "mean"),
    recall_std =("recall@k", "std"),
    precision_mean=("precision@k", "mean"),
    precision_std =("precision@k", "std"),
).reset_index()

print(f"\n=== Multi-seed stability (n={len(SEEDS)} seeds) ===")
print(agg.round(4).to_string(index=False))

### Stability table for the thesis

The row-per-k block below is publication-ready: it shows mean recall ± std and
a compact "mean ± std" column that reads naturally in a report table.

In [ ]:
thesis_table = pd.DataFrame({
    "k":              agg["k"].astype(int),
    "Recall@k":       [f"{m:.3f} ± {s:.3f}" for m, s in zip(agg["recall_mean"], agg["recall_std"])],
    "Precision@k":    [f"{m:.4f} ± {s:.4f}" for m, s in zip(agg["precision_mean"], agg["precision_std"])],
})
print(thesis_table.to_string(index=False))
thesis_table.to_csv("stability_results.csv", index=False)
print("\nSaved: stability_results.csv")

### Stability plot: recall curve with confidence band

The shaded band is ±1 standard deviation across the 5 seeds. A tight band means
the method is stable regardless of the random draw of pseudo-negatives.

In [ ]:
ks_smooth = list(range(50, 6001, 50))
seed_curves = []
for seed in SEEDS:
    scores, is_holdout = run_bagging_pu(X, y, seed)
    curve = precision_recall_at_k(scores, is_holdout, ks_smooth)["recall@k"].values
    seed_curves.append(curve)
seed_curves = np.array(seed_curves)     # (n_seeds, len(ks_smooth))

mean_curve = seed_curves.mean(axis=0)
std_curve  = seed_curves.std(axis=0)

plt.figure(figsize=(8, 5))
plt.plot(ks_smooth, mean_curve, color="C1", label=f"Bagging-PU (mean of {len(SEEDS)} seeds)")
plt.fill_between(ks_smooth, mean_curve - std_curve, mean_curve + std_curve,
                 color="C1", alpha=0.25, label="±1 std")
plt.xlabel("k (top-ranked prospects)")
plt.ylabel("Recall@k on held-out positives")
plt.title(f"Bagging-PU stability across {len(SEEDS)} random seeds")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("stability_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: stability_curve.png")

## 5. Experiment B: Elkan-Noto vs Bagging-PU

Same held-out validation set (single seed = 42, for fair per-method comparison), two methods.

In [ ]:
COMPARISON_SEED = 42
print(f"Running Bagging-PU (seed={COMPARISON_SEED})...")
pu_scores, is_holdout = run_bagging_pu(X, y, COMPARISON_SEED)

print(f"Running Elkan-Noto (seed={COMPARISON_SEED})...")
en_scores, is_holdout_en, c = run_elkan_noto(X, y, COMPARISON_SEED)
print(f"Estimated c = P(labeled | positive) = {c:.4f}")

# The two methods share the same held-out mask under the same seed.
assert (is_holdout == is_holdout_en).all()

pu_metrics = precision_recall_at_k(pu_scores, is_holdout, K_VALUES)
en_metrics = precision_recall_at_k(en_scores, is_holdout, K_VALUES)

comparison = pd.DataFrame({
    "k":                K_VALUES,
    "PU Recall@k":      pu_metrics["recall@k"].values.round(4),
    "EN Recall@k":      en_metrics["recall@k"].values.round(4),
    "PU hits":          pu_metrics["hits"].values,
    "EN hits":          en_metrics["hits"].values,
    "diff (PU - EN)":   (pu_metrics["recall@k"] - en_metrics["recall@k"]).round(4).values,
})
print(f"\n=== Method comparison at seed={COMPARISON_SEED} ===")
print(comparison.to_string(index=False))
comparison.to_csv("method_comparison.csv", index=False)
print("\nSaved: method_comparison.csv")

### Method comparison plot

In [ ]:
pu_curve = precision_recall_at_k(pu_scores, is_holdout, ks_smooth)["recall@k"].values
en_curve = precision_recall_at_k(en_scores, is_holdout, ks_smooth)["recall@k"].values

plt.figure(figsize=(8, 5))
plt.plot(ks_smooth, pu_curve, label="Bagging-PU", color="C1", linewidth=2)
plt.plot(ks_smooth, en_curve, label="Elkan-Noto",  color="C2", linewidth=2)
plt.xlabel("k (top-ranked prospects)")
plt.ylabel("Recall@k on held-out positives")
plt.title(f"Bagging-PU vs Elkan-Noto (seed={COMPARISON_SEED})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("method_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: method_comparison.png")

## 6. Summary and interpretation

**Experiment A (stability)** tells you how much noise sits under the headline numbers.
If std is small (~2 pts or less) at the useful k values, the pipeline is reproducible
and the reported gains are real. A large std (>5 pts) would be a warning that a single
run's numbers happened to be lucky (or unlucky) and shouldn't be trusted alone.

**Experiment B (Elkan-Noto)** shows whether the method choice was pivotal or interchangeable.
If Bagging-PU wins by a clear margin at every k, you can defend it as the better approach.
If Elkan-Noto ties or wins, that itself is a legitimate finding — the two methods encode
different assumptions about the unlabeled pool (Elkan-Noto assumes SCAR: Selected Completely
At Random; Bagging-PU is more robust to violations of that).

Both experiments produce CSV outputs (`stability_results.csv`, `method_comparison.csv`)
and PNG figures ready to drop into the thesis.

In [ ]:
print("Experiments complete. Files produced:")
print("  stability_results.csv     -- multi-seed Recall@k / Precision@k table")
print("  stability_curve.png       -- recall curve with ±1 std confidence band")
print("  method_comparison.csv     -- Bagging-PU vs Elkan-Noto side by side")
print("  method_comparison.png     -- overlaid recall curves for both methods")